# Environmental Data Science — Week 03
## Student version

Topics: log transformation, scaling, categorical encoding, and smoothed target encoding.

Complete the **8 exercises** by replacing each `____`. Some exercises contain more than one blank. Run cells from top to bottom. Cells containing blanks will fail until completed.

The setup and plotting code are provided. No answer cells or saved outputs are included.

**Modeling note:** These are classroom demonstrations on a single dataset. For predictive modeling, split the data first, fit preprocessing on training data only, and use out-of-fold target encoding for the training rows. Do not use validation/test targets to compute encodings.

## 1. Setup

In [ ]:
import os
os.chdir("/content")

In [ ]:
!rm -rf /content/Data
!git clone https://github.com/KU-EBL/Data.git

In [ ]:
import os
os.chdir("/content/Data/EDS")
os.listdir()

## 2. Load the data

Read `09_preprocessing_practice.csv` from `/content/Data/EDS`. Required columns: `nitrate_mg_L`, `temperature_C`, `rainfall_mm`, `land_use`, and `risk_level`.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("09_preprocessing_practice.csv")
df.head()

## 3. Original distribution

Compare the histogram and kernel density estimate (KDE).

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

x = df["nitrate_mg_L"].dropna()

# Visualize the original distribution
plt.figure(figsize=(6, 4))
plt.hist(x, bins=8, density=True, alpha=0.6)

kde = gaussian_kde(x)
x_range = np.linspace(x.min(), x.max(), 200)
plt.plot(x_range, kde(x_range), linewidth=2)

plt.xlabel("Nitrate (mg/L)")
plt.ylabel("Density")
plt.title("Original distribution with KDE")

plt.show()

## 4. Log transformation

Apply the natural logarithm of one plus each nitrate value. Compare the shape with the original distribution.

### Exercise 1

Complete the NumPy function for ln(1 + x).

In [ ]:
df["nitrate_log"] = np.____(df["nitrate_mg_L"])

x = df["nitrate_log"].dropna()

# Visualize the log-transformed distribution
plt.figure(figsize=(6, 4))
plt.hist(x, bins=8, density=True, alpha=0.6)

kde = gaussian_kde(x)
x_range = np.linspace(x.min(), x.max(), 200)
plt.plot(x_range, kde(x_range), linewidth=2)

plt.xlabel("ln(1 + nitrate)")
plt.ylabel("Density")
plt.title("Log-transformed distribution with KDE")

plt.show()

## 5. Scaling

Use temperature and rainfall to compare min–max normalization and standardization.

In [ ]:
df = pd.read_csv("09_preprocessing_practice.csv")
df[["temperature_C", "rainfall_mm"]]

### Exercise 2

Create a min–max scaler.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
minmax_scaler = ____()
normalized_data = minmax_scaler.fit_transform(
    df[["temperature_C", "rainfall_mm"]]
)
display(normalized_data)

### Exercise 3

Create a scaler for zero mean and unit standard deviation.

In [ ]:
from sklearn.preprocessing import StandardScaler
standard_scaler = ____()
standardized_data = standard_scaler.fit_transform(
    df[["temperature_C", "rainfall_mm"]]
)
display(standardized_data)

## 6. One-hot encoding

Create a separate indicator column for each land-use category.

In [ ]:
df

### Exercise 4

Complete the pandas function that creates one-hot columns.

In [ ]:
onehot = pd.____(df, columns = ['land_use'], dtype = int)
onehot

## 7. Ordinal encoding

Preserve the order of risk levels: Low < Medium < High.

### Exercise 5

Fill in the mapping dictionary for Low = 0, Medium = 1, High = 2.

In [ ]:
df["risk_level_encoded"] = df["risk_level"].map(____)
df

## 8. Target encoding

Replace each land-use category with its mean encoded risk level. This averages the assigned ordinal scores.

In [ ]:
df

### Exercise 6

Choose the aggregation statistic for target mean encoding.

In [ ]:
df["land_use_class"] = df.groupby("land_use")["risk_level_encoded"].transform(____)
df

## 9. Smoothed target encoding

Combine the category mean with the overall mean. The parameter `weight` corresponds to m in the slides.

Smoothed mean = (n × category mean + m × overall mean) / (n + m).

### Exercises 7–8

7. Calculate the overall target mean.
8. Complete the numerator and denominator of the smoothing formula.

In [ ]:
def weighted_target_encoding(df, cat_col, target_col, weight):
    """
    Parameters
    ----------
    df : pandas.DataFrame
        Input data frame.

    cat_col : str
        Name of the categorical column

    target_col : str
        Name of the target column

    weight : float
        Smoothing weight.
        In this example, the value is set to 2.
    """

    # Calculate the overall target mean
    overall_mean = ____

    # Calculate the count and mean for each category
    agg = df.groupby(cat_col)[target_col].agg(["count", "mean"])

    counts = agg["count"]   # Number of observations in each category
    means = agg["mean"]     # Mean target value for each category

    # Calculate the smoothed weighted target encoding
    smooth_mean = (
        ____
    ) / (
        ____
    )

    # Convert the encoded values into a dictionary
    encoding_mapping = smooth_mean.to_dict()

    # Add the encoded column to the original data frame
    df[cat_col + "_encoded"] = df[cat_col].map(
        encoding_mapping
    )

    return encoding_mapping, df

In [ ]:
weighted, df_new= weighted_target_encoding(df, "land_use", "risk_level_encoded", 2)

In [ ]:
weighted

In [ ]:
df_new

## Review questions

1. How does the log transformation change the distribution?
2. How do normalization and standardization differ?
3. Why is the risk-level mapping ordinal encoding?
4. What happens to smoothed encodings as m increases?